신입생 충원현황 계열확장 완료된 파일과 원래 분류 되어있는 등록금 파일의  공통 (기준년도, 학교, 대계열) 조합 수: 6,080개, 공통 학교 수: 202개
그에 맞춰 모든 연도에 존재하는 공통 조합, 공통학교에 해당하는 컬럼 데이터만 남기고 두 피처 파일을 병합함.
그에 따라 원래 설립 구분, 지역 등이 없던 등록금 데이터도 신입생 충원현황 데이터 옆에 컬럼이 되어 구분이 가능해짐.

In [ ]:
# 1. 필요한 라이브러리 불러오기
import pandas as pd


In [ ]:
# 2. 파일 불러오기
df_enrollment = pd.read_csv("/content/신입생_충원_현황(2014~2023)_충원율피처_정정본.csv")
df_register = pd.read_csv("/content/학과별_등록금_최종.csv")


In [ ]:
#  3. 연도별 공통 (학교, 대계열) 조합 구하기
school_major_per_year_enr = df_enrollment.groupby('기준년도')[['학교', '대계열']].apply(lambda x: set(tuple(row) for row in x.to_numpy()))
common_school_major_enr = set(school_major_per_year_enr.iloc[0])
for s in school_major_per_year_enr[1:]:
    common_school_major_enr &= s

school_major_per_year_reg = df_register.groupby('기준연도')[['학교명', '대계열']].apply(lambda x: set(tuple(row) for row in x.to_numpy()))
common_school_major_reg = set(school_major_per_year_reg.iloc[0])
for s in school_major_per_year_reg[1:]:
    common_school_major_reg &= s

common_school_major_both = common_school_major_enr & common_school_major_reg


In [ ]:
#  4. 신입생 충원율 데이터에서 공통 조합 필터링 및 정렬
df_enrollment['조합'] = list(zip(df_enrollment['학교'], df_enrollment['대계열']))
df_enrollment_common = df_enrollment[df_enrollment['조합'].isin(common_school_major_both)]
df_enrollment_common_sorted = df_enrollment_common.sort_values(by=['기준년도', '학교']).reset_index(drop=True)


In [ ]:
#  5. 등록금 데이터에서 공통 조합 필터링 및 평균 등록금 추출
df_register['조합'] = list(zip(df_register['학교명'], df_register['대계열']))
df_register_common = df_register[df_register['조합'].isin(common_school_major_both)]
등록금_매핑 = df_register_common.groupby(['학교명', '대계열'])['등록금'].mean().reset_index()
등록금_매핑.columns = ['학교', '대계열', '등록금']


In [ ]:
#  6. 등록금 컬럼 병합
df_final = df_enrollment_common_sorted.merge(등록금_매핑, on=['학교', '대계열'], how='left')
df_final = df_final.drop(columns=['조합'])  # 중간 컬럼 제거


In [ ]:
# 7. 저장
df_final.to_csv("/content/신입생_충원현황_공통조합_등록금추가.csv", index=False, encoding='utf-8-sig')
df_final.head()
